In [28]:
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import interact, IntSlider

# =====================================================
# Parameters
# =====================================================
true_mean = 10
true_sd = 5
max_N = 500

# =====================================================
# Interactive Function
# =====================================================
def update_plot(N, seed):

    # -------------------------------------------------
    # Generate signal
    # -------------------------------------------------
    np.random.seed(seed)

    x = np.random.normal(
        loc=true_mean,
        scale=true_sd,
        size=max_N
    )

    samples = np.arange(1, max_N + 1)

    # -------------------------------------------------
    # Running statistics
    # -------------------------------------------------
    sample_numbers = np.arange(1, N + 1)

    running_mean = []
    running_sd_N = []
    running_sd_Nm1 = []
    running_sd_known_mean = []

    for n in sample_numbers:

        subset = x[:n]

        # Mean is valid at N=1
        running_mean.append(
            np.mean(subset)
        )

        # SD undefined at N=1
        if n == 1:

            running_sd_N.append(np.nan)
            running_sd_Nm1.append(np.nan)
            running_sd_known_mean.append(np.nan)

        else:

            running_sd_N.append(
                np.std(subset, ddof=0)
            )

            running_sd_Nm1.append(
                np.std(subset, ddof=1)
            )

            running_sd_known_mean.append(
                np.sqrt(
                    np.mean(
                        (subset - true_mean) ** 2
                    )
                )
            )

    running_mean = np.array(running_mean)

    running_sd_N = np.array(running_sd_N)
    running_sd_Nm1 = np.array(running_sd_Nm1)
    running_sd_known_mean = np.array(
        running_sd_known_mean
    )

    # -------------------------------------------------
    # Errors
    # -------------------------------------------------
    sd_error_N = (
        running_sd_N - true_sd
    )

    sd_error_Nm1 = (
        running_sd_Nm1 - true_sd
    )

    sd_error_known_mean = (
        running_sd_known_mean - true_sd
    )

    # -------------------------------------------------
    # Theoretical bias curve
    # -------------------------------------------------
    theoretical_sd_N = np.full(
        len(sample_numbers),
        np.nan
    )

    valid = sample_numbers >= 2

    theoretical_sd_N[valid] = (
        true_sd *
        np.sqrt(
            (sample_numbers[valid] - 1)
            / sample_numbers[valid]
        )
    )

    # -------------------------------------------------
    # Create figure
    # -------------------------------------------------
    fig, ax = plt.subplots(
        4,
        1,
        figsize=(12, 18)
    )

    # =================================================
    # Plot 1 - Signal
    # =================================================
    ax[0].plot(
        samples[:N],
        x[:N],
        'o-',
        label='Random Signal'
    )

    ax[0].axhline(
        true_mean,
        color='red',
        linestyle='--',
        linewidth=2,
        label=f'True Mean = {true_mean}'
    )

    ax[0].set_title(
        f'Random Signal (First {N} Samples)'
    )

    ax[0].set_xlabel(
        'Sample Number'
    )

    ax[0].set_ylabel(
        'Amplitude'
    )

    ax[0].grid(True)
    ax[0].legend()

    # =================================================
    # Plot 2 - Running Mean
    # =================================================
    ax[1].plot(
        sample_numbers,
        running_mean,
        linewidth=2,
        label='Sample Mean'
    )

    ax[1].axhline(
        true_mean,
        color='red',
        linestyle='--',
        linewidth=2,
        label=f'True Mean = {true_mean}'
    )

    ax[1].set_title(
        'Convergence of Sample Mean'
    )

    ax[1].set_xlabel(
        'Number of Samples'
    )

    ax[1].set_ylabel(
        'Mean'
    )

    ax[1].grid(True)
    ax[1].legend()

    # =================================================
    # Plot 3 - SD Estimates
    # =================================================
    ax[2].plot(
        sample_numbers,
        running_sd_N,
        linewidth=2,
        label='SD using N'
    )

    ax[2].plot(
        sample_numbers,
        running_sd_Nm1,
        linewidth=2,
        label='SD using N-1'
    )

    ax[2].plot(
        sample_numbers,
        running_sd_known_mean,
        ':',
        linewidth=3,
        label='SD using Known Mean'
    )

    ax[2].plot(
        sample_numbers,
        theoretical_sd_N,
        'k--',
        linewidth=2,
        label='Expected SD using N'
    )

    ax[2].axhline(
        true_sd,
        color='red',
        linestyle='--',
        linewidth=2,
        label=f'True SD = {true_sd}'
    )

    ax[2].set_title(
        'Convergence of Standard Deviation Estimates'
    )

    ax[2].set_xlabel(
        'Number of Samples'
    )

    ax[2].set_ylabel(
        'Standard Deviation'
    )

    ax[2].grid(True)
    ax[2].legend()

    # =================================================
    # Plot 4 - SD Errors
    # =================================================
    ax[3].plot(
        sample_numbers,
        sd_error_N,
        linewidth=2,
        label='Error using N'
    )

    ax[3].plot(
        sample_numbers,
        sd_error_Nm1,
        linewidth=2,
        label='Error using N-1'
    )

    ax[3].plot(
        sample_numbers,
        sd_error_known_mean,
        ':',
        linewidth=3,
        label='Error using Known Mean'
    )

    ax[3].axhline(
        0,
        color='black',
        linestyle='--',
        linewidth=2
    )

    ax[3].set_title(
        'Standard Deviation Estimation Error'
    )

    ax[3].set_xlabel(
        'Number of Samples'
    )

    ax[3].set_ylabel(
        'Estimate - True SD'
    )

    ax[3].grid(True)
    ax[3].legend()

    # -------------------------------------------------
    # Keep all plots aligned
    # -------------------------------------------------
    for a in ax:
        a.set_xlim(1, N)

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------
    # Current values
    # -------------------------------------------------
    subset = x[:N]

    current_mean = np.mean(subset)

    current_sd_N = np.std(
        subset,
        ddof=0
    )

    current_sd_Nm1 = np.std(
        subset,
        ddof=1
    )

    current_sd_known = np.sqrt(
        np.mean(
            (subset - true_mean) ** 2
        )
    )

    print("=" * 55)
    print(f"Current Sample Size N = {N}")
    print(f"Random Seed = {seed}")
    print("=" * 55)

    print(f"True Mean              : {true_mean:.4f}")
    print(f"Sample Mean            : {current_mean:.4f}")

    print()

    print(f"True SD                : {true_sd:.4f}")
    print(f"SD using N             : {current_sd_N:.4f}")
    print(f"SD using N-1           : {current_sd_Nm1:.4f}")
    print(f"SD using Known Mean    : {current_sd_known:.4f}")

    print()

    print(f"Error using N          : {current_sd_N - true_sd:.4f}")
    print(f"Error using N-1        : {current_sd_Nm1 - true_sd:.4f}")
    print(f"Error using Known Mean : {current_sd_known - true_sd:.4f}")


# =====================================================
# Interactive Controls
# =====================================================
interact(
    update_plot,

    N=IntSlider(
        min=10,
        max=max_N,
        step=1,
        value=30,
        description='Samples'
    ),

    seed=IntSlider(
        min=0,
        max=9999,
        step=1,
        value=42,
        description='Seed'
    )
);

interactive(children=(IntSlider(value=30, description='Samples', max=500, min=10), IntSlider(value=42, descrip…

In [46]:
x = 3
N = 100000000
for i in range(N):
    x=x-0.0000003
print(f"x is {x}")

y = 300000000000
for i in range(N):
    y=y-0.0000003
print(f"y is {y}")

x is -26.99999999548774
y is 300000000000.0
